## What `-p` actually does under the hood

`-p` feels like magic, but it's two concrete mechanisms — and knowing them turns "the port doesn't work" from guesswork into a checklist. When you run `docker run -p 8080:80 nginx`, the daemon sets up:

1. **An `iptables` DNAT rule.** On Linux it inserts a rule in the `nat` table's `PREROUTING`/`DOCKER` chain that rewrites traffic destined for `host:8080` into `container-ip:80`. See it with `sudo iptables -t nat -L DOCKER -n`. This is the **fast path** — kernel-space, zero-copy.
2. **A userland proxy fallback (`docker-proxy`).** For traffic the DNAT rule can't catch (notably localhost-to-localhost, where `PREROUTING` is skipped), the daemon runs a small `docker-proxy` process listening on `host:8080` and forwarding to `container:80`. This is the **slow path** — a syscall round-trip per packet. You can turn it off with `"userland-proxy": false` in `daemon.json` (module 10); most teams leave it on.

Why this is worth knowing:

- **Debugging "I can't reach the port."** If a published port doesn't respond from the host, the iptables rule is the prime suspect — `sudo iptables -L -n -v` is your tool. A common cause is a firewall (ufw/firewalld) or a second daemon flushing Docker's chains.
- **The "port is already allocated" error.** If something on the host already listens on `8080`, the publish fails — `sudo ss -ltnp | grep 8080` finds the culprit.
- **Performance.** For most apps the cost is invisible; only very heavy network workloads need to measure `-p`'s impact (and that's often the reason to reach for `host` mode).

The point of the section: `-p` isn't opaque. It's a NAT rule plus an optional proxy, and both are inspectable — so port problems have a definite place to look.